In [1]:
import os
import pandas as pd
import re

# Основные пакеты
import langchain
import ollama
import chromadb

# НОВЫЕ ПУТИ ИМПОРТА (LangChain 0.1+)
from langchain_ollama import OllamaLLM  # Вместо langchain.llms import Ollama
from langchain_chroma import Chroma      # Вместо langchain.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings # Вместо langchain.embeddings

# Загрузчики и сплиттеры
from langchain_community.document_loaders import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Схемы и промпты
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.embeddings import Embeddings

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder

from langchain_core.runnables import RunnablePassthrough, RunnableLambda

c:\Users\Vyacheslave\Desktop\AI-assistant\rag_langchain_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Индексация

### Загрузка данных в Documents LangChain 

In [2]:
# 1. Загрузим данные
df = pd.read_csv(
    "data/archive/Air_condition_dataset.csv",
    encoding="cp1251",
    delimiter=";"
)

# 2. Выберем нужные столбцы и удалим строки с пустыми значениями
selected_columns = ["Наименование", "Подробное описание", "Характеристики"]
filtered_df = df[selected_columns].dropna()

# Функция для очистки текста
def clean_text(text):
    if not isinstance(text, str):
        return ""
    # Удаляем неразрывные пробелы и лишние символы
    text = re.sub(r'\s+', ' ', text)  # Заменяем все пробелы на один
    text = re.sub(r'[^\w\s.,!?-]', '', text)  # Удаляем нежелательные символы
    text = re.sub(r'&#\d+;', '', text)  # Удаляем HTML-сущности
    text = text.strip()  # Удаляем пробелы в начале и конце
    return text

# 3. Преобразуем в список документов LangChain
docs = []
for idx, row in filtered_df.iterrows():
    content = "\n".join(
        f"{col}: {clean_text(str(row[col]))}"
        for col in selected_columns
    )
    metadata = {"source": "Air_condition_dataset.csv", "row": idx}
    docs.append(Document(page_content=content, metadata=metadata))

# 4. Выведем первые 3 документа для проверки
for doc in docs[:10]:
    print(doc.page_content)
    print("---")

Наименование: LG ProCool 12
Подробное описание: Инверторные кондиционеры LG ProCool оснащены оригинальным компрессором LG и уникальным дизайном.Высочайшее качество компрессоров LG позволяет предлагать Вам Гарантию на компрессор 10 лет.Уникальный дизайн передней панели. Прозрачный пластик поверх белого пластикаСкрытый дисплей с мониторингом электропотребленияУправление и самодиагностика через встроенный WiFi модульPlasmaster Ionizer - Мощный поток ионов с целью очистки воздухаДополнительный угольный фильтр Тихая работа без вибрации6 режимов работы вентилятора внутреннего блока. Двойные автоматические жалюзи. Контроль направления воздуха с пультаУровень шума от 19 дБА. Спокойный сон при работающем кондиционереЗащитное покрытие теплообменника внешнего блока GOLD FIN1 год гарантии 43 2 года бесплатного гарантийного сервиса с заменой запчастей. 10 лет гарантии на компрессорУникальная схема монтажа. Удобно. Быстро. Полное прижимание внутреннего блока к стене.Голосовое управление посредством 

In [3]:
# Разделдение кажджой строки (документа) на чанки по 1000 символов
text_spliter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
splits = text_spliter.split_documents(docs)

In [4]:
# за 6 разделений уместилось на одном экземпляре всё три столбца
print(splits[0].page_content.replace('\n', ' '))
print("------------------------")
print(splits[1].page_content.replace('\n', ' '))
print("------------------------")
print(splits[2].page_content.replace('\n', ' '))
print("------------------------")
print(splits[3].page_content.replace('\n', ' '))
print("------------------------")
print(splits[4].page_content.replace('\n', ' '))
print("------------------------")
print(splits[5].page_content.replace('\n', ' '))


Наименование: LG ProCool 12
------------------------
Подробное описание: Инверторные кондиционеры LG ProCool оснащены оригинальным компрессором LG и уникальным дизайном.Высочайшее качество компрессоров LG позволяет предлагать Вам Гарантию на компрессор 10 лет.Уникальный дизайн передней панели. Прозрачный пластик поверх белого пластикаСкрытый дисплей с мониторингом электропотребленияУправление и самодиагностика через встроенный WiFi модульPlasmaster Ionizer - Мощный поток ионов с целью очистки воздухаДополнительный угольный фильтр Тихая работа без вибрации6 режимов работы вентилятора внутреннего блока. Двойные автоматические жалюзи. Контроль направления воздуха с пультаУровень шума от 19 дБА. Спокойный сон при работающем кондиционереЗащитное покрытие теплообменника внешнего блока GOLD FIN1 год гарантии 43 2 года бесплатного гарантийного сервиса с заменой запчастей. 10 лет гарантии на компрессорУникальная схема монтажа. Удобно. Быстро. Полное прижимание внутреннего блока к стене.Голосово

### Перевод в эмбеддинги

##### Модель просто для начала

In [5]:
EMBED_MODEL = "hf.co/Casual-Autopsy/snowflake-arctic-embed-l-v2.0-gguf:Q4_K_M"

OLLAMA_BASE_URL = "http://localhost:11434"

embeddings = OllamaEmbeddings(
    model = EMBED_MODEL,
    base_url = OLLAMA_BASE_URL 
)

embeddings

OllamaEmbeddings(model='hf.co/Casual-Autopsy/snowflake-arctic-embed-l-v2.0-gguf:Q4_K_M', dimensions=None, validate_model_on_init=False, base_url='http://localhost:11434', client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

##### Модель, подходящая для русского языка

In [6]:
# Использование другой модели под русский язык

# Базовая модель
base_embeddings = HuggingFaceEmbeddings(
    model_name="ai-forever/ru-en-RoSBERTa"
)

# Класс добавляет префиксы для документов и запросам пользователя, чтобы модель смогла их различать
class PrefixedEmbeddings(Embeddings):
    def __init__(self, base, query_prefix="", doc_prefix=""):
        self.base = base
        self.query_prefix = query_prefix
        self.doc_prefix = doc_prefix

    def embed_documents(self, texts):
        texts_prefixed = [self.doc_prefix + t for t in texts]
        return self.base.embed_documents(texts_prefixed)

    def embed_query(self, text):
        return self.base.embed_query(self.query_prefix + text)

embeddings = PrefixedEmbeddings(
    base_embeddings,
    query_prefix="search_query: ",
    doc_prefix="search_document: ",
)

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 7285.45it/s]
RobertaModel LOAD REPORT from: ai-forever/ru-en-RoSBERTa
Key                 | Status  | 
--------------------+---------+-
pooler.dense.weight | MISSING | 
pooler.dense.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
# Проба запросов
vec1 = embeddings.embed_query("инверторный")
vec2 = embeddings.embed_query("wi-fi")
vec3 = embeddings.embed_query("сплит-система")
vec4 = embeddings.embed_query("модель")

print(vec1,"\n--------------")
print(vec2,"\n--------------")
print(vec3,"\n--------------")
print(vec4,"\n--------------")


[-0.0248072799295187, -0.005064032971858978, -0.0097753144800663, 0.02458682283759117, 0.039205402135849, -0.02074367180466652, 0.019113214686512947, -0.0012510109227150679, -0.003844963386654854, -0.012528752908110619, 0.02152484469115734, -0.05994075909256935, -0.004409763030707836, 0.036709316074848175, -0.027766652405261993, -0.002522921422496438, -0.010645207017660141, -0.006370758172124624, -0.035052817314863205, -0.020558789372444153, 0.051021724939346313, 0.07072696089744568, 0.02782103791832924, 0.0016230305191129446, -0.004181722179055214, 0.02838868275284767, 0.03548445552587509, 0.04102778434753418, 0.024697214365005493, 0.018960420042276382, -0.023337289690971375, 0.02467912808060646, -0.03739530220627785, -0.020863519981503487, -0.030048338696360588, 0.008897044695913792, 0.01614656299352646, -0.040070392191410065, -0.06563647091388702, 0.012699599377810955, -0.013647545129060745, 0.017133234068751335, 0.02410406805574894, 0.035634204745292664, 0.0012682480737566948, -0.0

### Создание вектороного хранилища ChromaDB

##### Сделан механизм для проверки векторного хранилища, чтобы не перезаписывать всю ВБД-шку

In [8]:
from pathlib import Path
from langchain_community.vectorstores import Chroma

persist_directory = "./chroma_db"

if Path(persist_directory).exists():
    # Индекс уже есть – просто загружаем
    vectorstore = Chroma(
        embedding_function=embeddings,
        persist_directory=persist_directory,
    )
else:
    # Первый запуск – создаём индекс
    vectorstore = Chroma.from_documents(
        documents=splits,
        embedding=embeddings,
        persist_directory=persist_directory,
    )

C:\Users\Vyacheslave\AppData\Local\Temp\ipykernel_16232\1902338637.py:8: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


### Ретривер

##### Добавлены пару вариантов разынх видов ретриверов 

In [9]:
# 1. Базовый retriever с MMR (разнообразные документы)
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",  # вместо простого similarity
    search_kwargs={
        "k": 8,          # сколько документов вернуть в итоге
        "fetch_k": 32,   # из скольких кандидатов выбирать (больше = разнообразнее)
        # при желании можно добавить lambda_mult для тонкой настройки
        # lambda_mult – это параметр MMR, который задаёт баланс между релевантностью документов запросу и их разнообразием: чем ближе значение к 1, тем сильнее приоритет близости к запросу, чем ближе к 0 – тем важнее разнообразие результатов.
        # "lambda_mult": 0.8,
    },
)


# 2. Более строгий retriever с порогом релевантности
strict_retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",  # similarity_score_threshold – это режим поиска, при котором retriever возвращает только те документы, чья семантическая близость к запросу выше заданного порога, отсекая слабые и нерелевантные совпадения.
    search_kwargs={
        "score_threshold": 0.6,  # score_threshold – это порог релевантности (число от 0 до 1), который определяет минимальную допустимую степень семантической близости документа к запросу: чем выше значение, тем строже отбор, но более точные результаты.
        "k": 12,                 # максимум кандидатов, которые вообще рассматриваем
    },
)


# 3. Пример retriever только по FAQ (если в metadata есть source)
faq_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 6,
        # фильтр по метаданным – ключи должны совпадать с тем,
        # что ты кладёшь в doc.metadata при индексации
        "filter": {"source": {"$contains": "faq"}},
    },
)

### Шаблон промта

In [10]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Ты спецаилист в области систем кондиционирования"
        "Используй следующую информаци для ответа на вопрос"
        "Отвечай только на русском"
        "Отвечай только на те вопросы, которые связаны с облоастью твоей специальности"
        "Если вопрос не относится к твоей специальности вежливо уведоми об этом пользователя"
        "Если не можешь найти ответ на вопрос, вежливо уведоми пользователя о том, что не обладаешь знаниями об запрашиваемой информацией"
        "Если пользователя интресует какой-то товар, порекомендуй несколько вариантов, указывай бренд, модель и кратко о характеристиках, но постарайся выделить лучший, исходя из вопроса пользователя"
        "Всегда упоминай источник в формате из заголовка (Source и Page)"
            
    ),
    
    # Сохранение истории диалога
    MessagesPlaceholder("history"),
    
    (
        "human",
        "Контекст: {context}\n\n Вопрос: {question}"
        
    ),
    
])

prompt

ChatPromptTemplate(input_variables=['context', 'history', 'question'], input_types={'history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotat

### Генеративная модель (LLM)

In [13]:
OLLAMA_LLM_MODEL_BASE_URL =  "http://localhost:11434/v1"
LLM_MODEL = "mistral-nemo:latest"

llm = ChatOpenAI(
    api_key='None',
    base_url=OLLAMA_LLM_MODEL_BASE_URL,
    model=LLM_MODEL,
    
    # важные параметры для RAG
    temperature=0.4,      # меньше фантазии
    max_tokens=512,       # контролируем длину ответа
    top_p=0.9,
)

llm

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001C51739F410>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001C519B719D0>, root_client=<openai.OpenAI object at 0x000001C51714D890>, root_async_client=<openai.AsyncOpenAI object at 0x000001C519B72750>, model_name='mistral-nemo:latest', temperature=0.4, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='http://localhost:11434/v1', top_p=0.9, max_tokens=512)

### Проверка работы LLM

In [14]:
llm.invoke("Какие есть инверторные системы до 50 тыс. и самые тихие?")

AIMessage(content='Инверторные системы, также известные как инверторные кондиционеры, являются одними из самых энергоэффективных и бесшумных систем охлаждения на рынке. Ниже приведены некоторые из лучших инверторных систем до 50 тыс., а также наиболее тихие модели:\n\n1. Daikin FTXG75TVMA2 - это высокоэффективная инверторная система с рейтингом ENERGY STAR и шумовым уровнем всего 36 дБ(A) в режиме охлаждения. Эта модель идеально подходит для небольших офисов или жилых помещений.\n2. Fujitsu AOU5RLH - эта модель является одной из самых тихих инверторных систем на рынке, с шумовым уровнем всего 19 дБ(A) в режиме охлаждения. Он также имеет высокую энергоэффективность и идеально подходит для небольших офисов или домашнего использования.\n3. LG LS-H27RVN5 - эта модель является одной из самых популярных инверторных систем на рынке благодаря своей высокой энергоэффективности, тихой работе (шумовой уровень 28 дБ(A) в режиме охлаждения) и широкому диапазону мощностей.\n4. Samsung AR18HSV5000V/B

### Создание pipeline (пост-процессинг)

In [15]:
max_chars = 8000

def format_docs(docs):
    formatted = []
    total_len = 0

    for doc in docs:
        source = doc.metadata.get("source", "unknown_source")
        page = doc.metadata.get("page", None)

        header = f"Source: {source}"
        if page is not None:
            header += f" | Page: {page}"

        text = doc.page_content.strip()
        block = f"{header}\n{text}"

        # если следующий блок слишком раздует контекст – останавливаемся
        if total_len + len(block) > max_chars:
            break

        formatted.append(block)
        total_len += len(block)

    return "\n\n---\n\n".join(formatted)

In [17]:
def ensure_context(input_dict: dict) -> dict:
    """
    Если retriever не нашёл ничего полезного и контекст пустой,
    явно помечаем это в контексте, чтобы модель не фантазировала.
    """
    context = input_dict.get("context", "").strip()
    if not context:
        input_dict["context"] = (
            "Контекст пуст: ретривер не нашёл ни одного подходящего фрагмента. "
            "Если ответ важен, лучше явно сказать пользователю об этом."
        )
    return input_dict

rag_chain = (
    {
        "context": mmr_retriever | format_docs,
        "question": RunnablePassthrough(),
        "history": lambda _: [],  # пока истории нет – передаём пустой список
    }
    | RunnableLambda(ensure_context)   # защита от пустого контекста
    | prompt
    | llm
    | StrOutputParser()
).with_config(run_name="rag_chain")


### Тест всей RAG-цепи

In [18]:
rag_chain.invoke("Какие есть сплит-системы до 30 квадратных меторов от Hitachi?")

'Источник: Air_condition_dataset.csv\n\nНаименование: Сплит-система Hitachi R-Series RAS-F25GUHA\nПодробное описание: Сплит-система Hitachi R-Series с инверторным компрессором обеспечивает быстрый и эффективный обогрев или охлаждение помещений до 30 кв. м. Внутренний блок выполнен в строгом черном цвете, что идеально впишется в любой интерьер. Система оснащена фильтрами Active Carbon и Silver Ion для очистки воздуха от пыли, аллергенов и неприятных запахов. Функция 3D AUTO AIR позволяет равномерно распределять холодный или горячий воздух по всему помещению.\n\nХарактеристики:\n- Тип фреона: R410A\n- Холодопроизводительность: 2,5 кВт (охлаждение), 3,1 кВт (обогрев)\n- Класс энергоэффективности: A++\n- Максимальная длина трубопровода: 20 м\n\nНаименование: Сплит-система Hitachi R-Series RAS-F25JUHA\nПодробное описание: Сплит-система Hitachi R-Series с инверторным компрессором обеспечивает быстрый и эффективный обогрев или охлаждение помещений до 30 кв. м. Внутренний блок выполнен в строг

In [19]:
rag_chain.invoke("Какие есть инверторные системы до 50 тыс. и самые тихие?")

'Source: Air_condition_dataset.csv\n\nЕсли вам нужна инверторная сплит-система стоимостью до 50 тысяч рублей, я бы порекомендовал обратить внимание на следующую модель:\n\n1. Сплит-система Hisense ERA Classic A Wi-Fi AS-07HW4RLRKC00A\n\t* Характеристики:\n\t\t+ Энергоэффективность класса А\n\t\t+ Уровень шума внутреннего блока от 23,5 дБА на первой скорости вентилятора\n\t\t+ Четыре режима работы: охлаждение, обогрев, осушение и вентиляция\n\t\t+ Встроенный модуль Wi-Fi для управления через мобильное приложение\n\t* Цена: около 40 тысяч рублей\n\nЕсли тишина является приоритетом, то я бы порекомендовал следующую модель:\n\n1. Сплит-система QUATTROCLIMA LANTERNAQV-LA12WAEQN-LA12WAE\n\t* Характеристики:\n\t\t+ 7 скоростей вентилятора с уровнем шума от 21 дБ\n\t\t+ Повышенный класс сезонной энергоэффективности А4343 в режиме охлаждения и А43 в режиме обогрева\n\t\t+ Инверторные компрессоры для повышенной эффективности и надежности\n\t* Цена: около 45 тысяч рублей\n\nОба этих варианта пред